In [0]:
# Databricks Notebook: Introduction to Retrieval-Augmented Generation (RAG)

# AI Architect Guide: Teaching RAG from Basics (Topic 3)

# Step 1: Setup and Installations
# (Run this in a Databricks notebook cell)
!pip install --upgrade langchain langchain-community openai faiss-cpu



In [0]:
%pip install -q -U llama-index PyPDF2
dbutils.library.restartPython()

In [0]:
# Reduce the arrow batch size as our PDF can be big in memory
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 10)

In [0]:
# Copy from Workspace to DBFS
dbutils.fs.cp(
  "file:/Workspace/Shared/DatabricksAI2025-26/Training_Notebooks/animal_stories", 
  "dbfs:/tmp/animal_stories", 
  recurse=True
)


%md

# Preparing Data for RAG


**In this demo, we will focus on ingesting PDF documents as a source for our retrieval process.** First, we will ingest the PDF files and process them using self-managed vector search embeddings. 


**Learning Objectives:**

*By the end of this demo, you will be able to:*

* Split the data into chunks that are at least as small as the maximum context window of the LLM to be used later.

* Describe how to appropriately choose an embedding model.

* Compute embeddings for each of the chunks using a Databricks-managed embedding model.

* Use the chunking strategy to divide up the context information to be provided to a model.



## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**

%md
## Demo Overview

For this example, we will add research articles on GenerativeAI as PDFs from [Arxiv resources page](https://arxiv.org/abs/2309.07930) to our knowledge database.

<img src="https://files.training.databricks.com/images/genai/genai-as-01-rag-pdf-self-managed-0.png" style="float: right; width: 600px; margin-left: 10px">


Here are all the detailed steps:

- Use Spark to load the binary PDFs into our first table. 
- Use the `unstructured` library  to parse the text content of the PDFs.
- Use `llama_index` or `langchain` to split the texts into chunks.
- Compute embeddings for the chunks.
- Save our text chunks + embeddings in a Delta Lake table, ready for Vector Search indexing.


Databricks Lakehouse AI not only provides state of the art solutions to accelerate your AI and LLM projects but also to accelerate data ingestion and preparation at scale, including unstructured data like PDFs.

## Extract PDF Content as Text Chunks

As the first step, we need to ingest PDF files and divide the content into chunks. PDF files are already downloaded during the course step and stored in **datasets path**.

In [0]:
articles_path = "dbfs:/tmp/animal_stories"
table_name = "training_catalog.training_schema.pdf_raw_text"

# Read PDF files as binary
df = (
    spark.read.format("binaryfile")
    .option("recursiveFileLookup", "true")
    .load(articles_path)
)

# Save to table
df.write.mode("overwrite").saveAsTable(table_name)

# Display the dataframe
display(df)


In [0]:
%sql

select * from training_catalog.training_schema.pdf_raw_text

Let's view the content of one of the articles.

In [0]:
import io
from PyPDF2 import PdfReader
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# 1. Define a function to extract text from binary content
def extract_text_from_pdf_binary(binary_content: bytes) -> str:
    try:
        pdf_stream = io.BytesIO(binary_content)
        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            extracted_text += page.extract_text() or ""
        return extracted_text
    except Exception as e:
        return f"Error: {str(e)}"

# 2. Register UDF
extract_text_udf = udf(extract_text_from_pdf_binary, StringType())

# 3. Read your table
pdf_df = spark.table("training_catalog.training_schema.pdf_raw_text")

# 4. Apply the UDF to extract text
pdf_text_df = pdf_df.withColumn("extracted_text", extract_text_udf("content"))

# 5. Show the result
display(pdf_text_df.select("extracted_text"))


This looks great. We'll now wrap it with a `text_splitter` to avoid having too big pages and create a **Pandas UDF function** to easily scale that across multiple nodes.

**📌Note:** The pdf text isn't clean. To make it nicer, we could use a few extra LLM-based pre-processing steps, asking to remove unrelevant content like the list of chapters and to only keep the core text.

In [0]:
import io
import os
import pandas as pd 

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.core.utils import set_global_tokenizer
from transformers import AutoTokenizer
from typing import Iterator
from pyspark.sql.functions import col, udf, length, pandas_udf, explode
from PyPDF2 import PdfReader  # Updated import

# Define a function to split the text content into chunks
@pandas_udf("array<string>")
def read_as_chunk(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    # Set llama2 as tokenizer
    set_global_tokenizer(
      AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")
    )
    # Sentence splitter from llama_index to split on sentences
    splitter = SentenceSplitter(chunk_size=800, chunk_overlap=300)
    
    def extract_and_split(b):
        txt = extract_text_from_pdf_binary(b)  # Use custom function
        if txt is None:
            return []
        nodes = splitter.get_nodes_from_documents([Document(text=txt)])
        return [n.text for n in nodes]

    for x in batch_iter:
        yield x.apply(extract_and_split)

In [0]:
df_chunks = (df
                .withColumn("content", explode(read_as_chunk("content")))
                .selectExpr('path as pdf_name', 'content')
                )
display(df_chunks)

**💡 Chunking Overlap**: Review the content chunks and pay attention to the sentence overlap between chunks. This was defined with `SentenceSplitter` above.

## Embeddings with Foundation Model Endpoints

<img src="https://files.training.databricks.com/images/genai/genai-as-01-rag-pdf-self-managed-4.png" width="100%">

Foundation Models are provided by Databricks, and can be used out-of-the-box.

Databricks supports several endpoint types to compute embeddings or evaluate a model:
- A **foundation model endpoint**, provided by databricks (ex: llama3.1-70B, Mixtral-8-7B...)
- An **external endpoint**, acting as a gateway to an external model (ex: Azure OpenAI)
- A **custom**, fine-tuned model hosted on Databricks model service

Open the [Model Serving Endpoint page](/ml/endpoints) to explore and try the foundation models.

For this demo, we will use the foundation model `GTE` (embeddings) and `llama3.1-70B` (chat). <br/><br/>

![serving](files/images/generative-ai-solution-development-2.0.0/serving.png)

How to Use Foundation Model API
Before the compute the embeddings for the chunked text we created before, let's quickly go over how to use Foundation Model API.

🚨 Important: You will need Foundation Model API access for this section and the rest of the demo.

In [0]:
from mlflow.deployments import get_deploy_client


# gte-large-en Foundation models are available using the /serving-endpoints/databricks-gte-large-en/invocations api. 
deploy_client = get_deploy_client("databricks")

# NOTE: if you change your embedding model here, make sure you change it in the query step too
embeddings = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": ["What is Apache Spark?"]})
print(embeddings)

### Compute Chunking Embeddings

The last step is to now compute an embedding for all our documentation chunks. Let's create an udf to compute the embeddings using the foundation model endpoint.

**📌 Note:** This part would typically be setup as a production-grade job, running as soon as a new documentation page is updated.
This could be setup as a Delta Live Table pipeline to incrementally consume updates.

In [0]:
@pandas_udf("array<float>")
def get_embedding(contents: pd.Series) -> pd.Series:
    import mlflow.deployments
    deploy_client = mlflow.deployments.get_deploy_client("databricks")
    def get_embeddings(batch):
        # NOTE: this will fail if an exception is thrown during embedding creation (add try/except if needed) 
        response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": batch})
        return [e["embedding"] for e in response.data]

    # splitting the contents into batches of 150 items each, since the embedding model takes at most 150 inputs per request.
    max_batch_size = 150
    batches = [contents.iloc[i:i + max_batch_size] for i in range(0, len(contents), max_batch_size)]

    # process each batch and collect the results
    all_embeddings = []
    for batch in batches:
        all_embeddings += get_embeddings(batch.tolist())

    return pd.Series(all_embeddings)

In [0]:
import pyspark.sql.functions as F

df_chunk_emd = (df_chunks
                .withColumn("embedding", get_embedding("content"))
                .selectExpr("pdf_name", "content", "embedding")
                )
display(df_chunk_emd)

## Save Embeddings to a Delta Table

Now that the embeddings are ready, let's create a Delta table and store the embeddings in this table.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS training_catalog.training_schema.pdf_text_embeddings (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY,
  pdf_name STRING,
  content STRING,
  embedding ARRAY <FLOAT>
  -- NOTE: the table has to be CDC because VectorSearch is using DLT that is requiring CDC state
  ) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
embedding_table_name = f"training_Catalog.training_schema.pdf_text_embeddings"
df_chunk_emd.write.mode("overwrite").saveAsTable(embedding_table_name)

In [0]:
%sql

select * from training_Catalog.training_schema.pdf_text_embeddings


## Conclusion

In this demo, we demonstrated how to ingest and process documents for a RAG application. The first was to extract text chunks from PDF documents. Then, we created embeddings using a foundation model. This process includes setting up an endpoint and computing embeddings for the chunks. In the final step, we stored computed embeddings in a Delta table.